# Госпитальные_исходы: проверка обезличенного аналитического набора

Первичная сверка исходных медицинских таблиц была выполнена до экспорта. Исходные таблицы с идентификаторами в репозиторий не включены. Этот ноутбук проверяет опубликованную аналитическую матрицу и формирует рабочую копию.


In [ ]:
from pathlib import Path
import sys, os, json, hashlib, platform, importlib, datetime, tempfile
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "clinical-mortality-mpl"))
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "repository_support.py").exists())
if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from repository_support import initialize
OUT = initialize("vascular")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
BASE_SEED = 2026
VERSIONS = {m: importlib.import_module(m).__version__ for m in ["numpy", "pandas", "sklearn", "optuna", "shap", "catboost", "xgboost", "nbclient"]}
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def savejson(obj, name): (OUT / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str))
def figsave(name):
    plt.tight_layout(); plt.savefig(OUT / (name + ".png"), dpi=180, bbox_inches="tight"); plt.savefig(OUT / (name + ".svg"), bbox_inches="tight"); plt.show(); plt.close()
print("Working directory:", OUT, "Python:", platform.python_version(), "seed:", BASE_SEED, VERSIONS)


In [ ]:
meta = json.loads((OUT / "01_конфигурация.json").read_text())
D = pd.read_csv(OUT / "01_данные.csv")
assert sha(OUT / "01_данные.csv") == meta["data_sha256"]
assert list(D.columns) == ["row_id", "group_id"] + meta["features"] + ["hospital_death"]
assert len(D) == 548 and int(D.hospital_death.sum()) == 55
assert D.row_id.tolist() == list(range(len(D)))
assert set(D.hospital_death.unique()) == {0, 1}
assert D.group_id.nunique() == 511
display(pd.read_csv(OUT / "01_словарь.csv"))
display(D[meta["features"]].isna().sum().rename("missing"))
print("Data checked:", len(D), "rows;", int(D.hospital_death.sum()), "death marks")
